In [4]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')

from src.data import fetch_yahoo_data, merge_yahoo_datasets, fetch_weather_data, compute_rolling_weather, merge_price_weather

save_path = '/Users/wiktor/TIC/Spring2026-TIC/Ensemble_Wiktor'


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
# Fetch corn and soybean futures data
fetch_yahoo_data('CORN', 'corn', save_path=save_path)
fetch_yahoo_data('SOYB', 'soybean', save_path=save_path)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,close_soybean,high_soybean,low_soybean,open_soybean,volume_soybean
date,,,,,
2011-09-19,24.549999,24.549999,24.549999,24.549999,100
2011-09-20,25.080000,25.080000,24.660000,24.990000,1200
2011-09-21,24.340000,25.110001,24.340000,24.959999,4400
2011-09-22,24.000000,24.000000,24.000000,24.000000,100
2011-09-23,23.000000,24.430000,23.000000,24.430000,300
...,...,...,...,...,...
2026-03-05,24.000000,24.049999,23.809999,23.879999,57900
2026-03-06,24.410000,24.450001,24.250000,24.299999,266500
2026-03-09,24.219999,24.660000,24.180000,24.639999,359300


In [14]:
# merge yahoo datasets
merge_yahoo_datasets(['corn', 'soybean'], 'merged_corn_soybean', save_path=save_path)

,close_corn,high_corn,low_corn,open_corn,volume_corn,close_soybean,high_soybean,low_soybean,open_soybean,volume_soybean
date,,,,,,,,,,
2011-09-19,45.650002,45.919998,45.000000,45.459999,114300,24.549999,24.549999,24.549999,24.549999,100
2011-09-20,45.540001,46.720001,45.369999,46.110001,117200,25.080000,25.080000,24.660000,24.990000,1200
2011-09-21,44.799999,46.040001,44.750000,45.900002,83300,24.340000,25.110001,24.340000,24.959999,4400
2011-09-22,42.990002,44.669998,42.650002,43.959999,352500,24.000000,24.000000,24.000000,24.000000,100
2011-09-23,42.450001,43.389999,42.400002,42.990002,158800,23.000000,24.430000,23.000000,24.430000,300
...,...,...,...,...,...,...,...,...,...,...
2026-03-05,18.190001,18.209999,17.900000,17.900000,389600,24.000000,24.049999,23.809999,23.879999,57900
2026-03-06,18.520000,18.559999,18.299999,18.370001,943400,24.410000,24.450001,24.250000,24.299999,266500
2026-03-09,18.200001,18.730000,18.160000,18.620001,1105000,24.219999,24.660000,24.180000,24.639999,359300


In [ ]:
# US Corn Belt

LOCATIONS_US = {
    "central_iowa": (42.0, -93.5),        # Heart of corn/soy production
    "central_illinois": (40.1, -89.4),     # Major corn/soy state
    "central_indiana": (40.0, -86.2),      # Eastern corn belt
}

# Brazil soy regions (optional, for soybean-specific weather)
LOCATIONS_BRAZIL = {
    "mato_grosso": (-12.6, -55.5),         # Largest soy-producing state
    "parana": (-24.0, -51.5),              # Second-largest soy state
}

weather = fetch_weather_data(
    locations={**LOCATIONS_US, **LOCATIONS_BRAZIL},
    start_date="2005-01-01",
    save_path="../data/weather.csv",
)

Processing 'central_iowa': 42.00°N, -93.54°W
Processing 'central_illinois': 40.11°N, -89.37°W
Processing 'central_indiana': 40.04°N, -86.22°W
Processing 'mato_grosso': -12.62°N, -55.47°W
Processing 'parana': -24.01°N, -51.44°W

Saved to: ../data/weather.csv


In [2]:
# Transform the data
weather_rolled = compute_rolling_weather(
    "../data/weather.csv",
    window=30,
    save_path="../data/weather_rolled.csv",
)

Window:          30 days
Input columns:   75
Output columns:  75
Date range:      2005-01-30 00:00:00 to 2026-03-11 00:00:00
Valid rows:      7711 / 7711
Saved to:        ../data/weather_rolled.csv


In [7]:
# Merge weather and price data
df = merge_price_weather(
    "../data/merged_corn_soybean.csv",
    "../data/weather_rolled.csv",
    save_path="../data/full_dataset.csv",
)

Price columns:   10
Weather columns: 75
Merged shape:    (3640, 85)
Date range:      2011-09-19 00:00:00 to 2026-03-11 00:00:00
Missing values:  0
Saved to:        ../data/full_dataset.csv
